In [20]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mrwellsdavid/unsw-nb15/UNSW_NB15_testing-set.csv
/kaggle/input/datasets/mrwellsdavid/unsw-nb15/UNSW-NB15_1.csv
/kaggle/input/datasets/mrwellsdavid/unsw-nb15/UNSW_NB15_training-set.csv
/kaggle/input/datasets/mrwellsdavid/unsw-nb15/UNSW-NB15_LIST_EVENTS.csv
/kaggle/input/datasets/mrwellsdavid/unsw-nb15/UNSW-NB15_4.csv
/kaggle/input/datasets/mrwellsdavid/unsw-nb15/UNSW-NB15_3.csv
/kaggle/input/datasets/mrwellsdavid/unsw-nb15/UNSW-NB15_2.csv
/kaggle/input/datasets/mrwellsdavid/unsw-nb15/NUSW-NB15_features.csv


In [21]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from xgboost import XGBClassifier
import matplotlib.pyplot as plt


**Load Feature Metadata**

In [22]:
features = pd.read_csv(
    "/kaggle/input/datasets/mrwellsdavid/unsw-nb15/NUSW-NB15_features.csv",
    encoding="latin1"
)

features.head()

,No.,Name,Type,Description
0,1,srcip,nominal,Source IP address
1,2,sport,integer,Source port number
2,3,dstip,nominal,Destination IP address
3,4,dsport,integer,Destination port number
4,5,proto,nominal,Transaction protocol


**Validate Feature Names**

In [23]:
column_names = features['Name'].tolist()

print(column_names[-5:])
print(len(column_names))

['ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat', 'Label']
49


In [24]:
print(column_names)
print(len(column_names))

['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Stime', 'Ltime', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat', 'Label']
49


**Load UNSW-NB15 Dataset**

In [25]:
files = [
    "/kaggle/input/datasets/mrwellsdavid/unsw-nb15/UNSW-NB15_1.csv",
    "/kaggle/input/datasets/mrwellsdavid/unsw-nb15/UNSW-NB15_2.csv",
    "/kaggle/input/datasets/mrwellsdavid/unsw-nb15/UNSW-NB15_3.csv",
    "/kaggle/input/datasets/mrwellsdavid/unsw-nb15/UNSW-NB15_4.csv"
]

dfs = []

for file in files:
    temp_df = pd.read_csv(
        file,
        header=None,
        names=column_names
    )
    dfs.append(temp_df)

df = pd.concat(dfs, ignore_index=True)

/tmp/ipykernel_58/728757038.py:11: DtypeWarning: Columns (1,3,47) have mixed types. Specify dtype option on import or set low_memory=False.
  temp_df = pd.read_csv(
/tmp/ipykernel_58/728757038.py:11: DtypeWarning: Columns (3,39,47) have mixed types. Specify dtype option on import or set low_memory=False.
  temp_df = pd.read_csv(


**Dataset Shape**

In [26]:
print(df.shape)

(2540047, 49)


**Missing Value Analysis**

In [27]:
df.isnull().sum().sort_values(ascending=False)

attack_cat          2218764
is_ftp_login        1429879
ct_flw_http_mthd    1348145
sport                     0
proto                     0
state                     0
dstip                     0
dsport                    0
dbytes                    0
sttl                      0
dttl                      0
sloss                     0
dloss                     0
service                   0
dur                       0
sbytes                    0
srcip                     0
Spkts                     0
Dload                     0
Sload                     0
Dpkts                     0
dtcpb                     0
swin                      0
dwin                      0
stcpb                     0
res_bdy_len               0
Sjit                      0
Djit                      0
Stime                     0
Ltime                     0
smeansz                   0
dmeansz                   0
trans_depth               0
tcprtt                    0
Dintpkt                   0
Sintpkt             

**Handle Missing Attack Labels**

In [28]:
df['attack_cat'] = df['attack_cat'].fillna('Normal')

**FTP Login Preprocessing**

In [29]:
df['is_ftp_login'] = df['is_ftp_login'].fillna(0)

**HTTP Method Preprocessing**

In [30]:
df['ct_flw_http_mthd'] = df['ct_flw_http_mthd'].fillna(0)

**Remove Invalid Records**

In [31]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

df.dropna(inplace=True)

**Verify Data Quality**

In [32]:
df.isnull().sum().sort_values(ascending=False)

srcip               0
sport               0
dstip               0
dsport              0
proto               0
state               0
dur                 0
sbytes              0
dbytes              0
sttl                0
dttl                0
sloss               0
dloss               0
service             0
Sload               0
Dload               0
Spkts               0
Dpkts               0
swin                0
dwin                0
stcpb               0
dtcpb               0
smeansz             0
dmeansz             0
trans_depth         0
res_bdy_len         0
Sjit                0
Djit                0
Stime               0
Ltime               0
Sintpkt             0
Dintpkt             0
tcprtt              0
synack              0
ackdat              0
is_sm_ips_ports     0
ct_state_ttl        0
ct_flw_http_mthd    0
is_ftp_login        0
ct_ftp_cmd          0
ct_srv_src          0
ct_srv_dst          0
ct_dst_ltm          0
ct_src_ ltm         0
ct_src_dport_ltm    0
ct_dst_spo

**Feature and Target Separation**

In [33]:
X = df.drop(
    columns=[
        'srcip',
        'dstip',
        'Stime',
        'Ltime',
        'attack_cat',
        'Label'
    ],
    errors='ignore'
)

y = df['attack_cat']

**Save Categorical Encoders**

In [37]:
categorical_cols = X.select_dtypes(
    include=['object']
).columns


encoders = {}
X_copy = X.copy()
for col in categorical_cols:
    le = LabelEncoder()
    le.fit(
        X_copy[col].astype(str)
    )
    encoders[col] = le
joblib.dump(
    encoders,
    "feature_encoders.pkl"
)
print(encoders.keys())

dict_keys(['sport', 'dsport', 'proto', 'state', 'service', 'ct_ftp_cmd'])


**Encode Target Labels**

In [38]:
target_encoder = LabelEncoder()

y = target_encoder.fit_transform(y)

**Identify Categorical Features**

In [39]:
categorical_cols = X.select_dtypes(include=['object']).columns

print(categorical_cols)

Index(['sport', 'dsport', 'proto', 'state', 'service', 'ct_ftp_cmd'], dtype='object')


**Encode Categorical Variables**

In [40]:
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

**Memory Usage Analysis**

In [41]:
# Check memory usage of feature matrix
memory_usage_gb = X.memory_usage(deep=True).sum() / 1024**3
print(f"Feature Matrix Memory Usage: {memory_usage_gb:.2f} GB")

Feature Matrix Memory Usage: 0.81 GB


**Memory Optimization**

In [42]:
# Reduce memory usage
for col in X.select_dtypes(include=['float64']).columns:
    X[col] = X[col].astype('float32')

for col in X.select_dtypes(include=['int64']).columns:
    X[col] = X[col].astype('int32')

optimized_memory_gb = X.memory_usage(deep=True).sum() / 1024**3

print(f"Optimized Memory Usage: {optimized_memory_gb:.2f} GB")

Optimized Memory Usage: 0.41 GB


**Train/Test Split**

In [43]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

Training Shape: (2032037, 43)
Testing Shape: (508010, 43)


**Handle Class Imbalance**

In [46]:
smote = SMOTE(random_state=42)

X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

**Train Baseline XGBoost Model**

In [47]:
model = XGBClassifier(
    n_estimators=50,
    max_depth=6,
    learning_rate=0.1,
    objective='multi:softprob',
    eval_metric='mlogloss',
    tree_method='hist',
    n_jobs=-1
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=50, n_jobs=-1,
              num_parallel_tree=None, ...)

**Train SMOTE-Enhanced Model**

In [ ]:
model_res = XGBClassifier(
    n_estimators=50,
    max_depth=6,
    learning_rate=0.1,
    objective='multi:softprob',
    eval_metric='mlogloss',
    tree_method='hist',
    n_jobs=-1
)

model_res.fit(X_resampled, y_resampled)

**Baseline Model Predictions**

In [ ]:
y_pred = model.predict(X_test)

**SMOTE Model Predictions**

In [ ]:
y_pred_resampled = model_res.predict(X_test)

**Performance of Baseline model**

In [ ]:
print(classification_report(y_test, y_pred))

weighted_f1 = f1_score(
    y_test,
    y_pred,
    average='weighted'
)

print(f"Weighted F1 Score: {weighted_f1:.4f}")

**Performance of SMOTE model**

In [ ]:
print(classification_report(y_test, y_pred_resampled))

weighted_f1 = f1_score(
    y_test,
    y_pred,
    average='weighted'
)

print(f"Weighted F1 Score: {weighted_f1:.4f}")

**Feature Importance Analysis**

In [ ]:
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': model_res.feature_importances_
})

importance_df = importance_df.sort_values(
    by='importance',
    ascending=False
)

importance_df.head(20)

**Feature Importance Visualization**

In [ ]:
top_features = importance_df.head(20)

plt.figure(figsize=(12,8))

plt.barh(
    top_features['feature'],
    top_features['importance']
)

plt.gca().invert_yaxis()

plt.title("Top 20 Important Features")

plt.show()

**Save Trained Artifacts**

In [ ]:
joblib.dump(model, "xgb_base_model.pkl")
joblib.dump(model_res, "xgb_smote_model.pkl")
joblib.dump(target_encoder, "target_encoder.pkl")
joblib.dump(list(X.columns), "feature_columns.pkl")

**Export Feature Importance**

In [ ]:
importance_df.to_csv(
    "feature_importance.csv",
    index=False
)